# Topic 4: CI/CD for Spark with GitHub Actions

> **Phase 7 → Week 15 → Topic 4**

---

## What Is CI/CD for Data Pipelines?

```
CI (Continuous Integration):   every PR runs lint + tests automatically
CD (Continuous Deployment):    merged code is automatically deployed to staging/prod
```

For Spark pipelines, CI/CD means:
- Every code change runs `pytest` (unit tests + integration tests)
- Code is linted (`ruff` for Python, type-checked with `mypy`)
- Scripts are packaged and uploaded to S3 automatically
- EMR/Glue jobs are triggered with the new code on merge to main

---

## Interview Cheat Sheet

**Q: How do you implement CI/CD for a Spark ETL pipeline?**
> GitHub Actions (or GitLab CI/Jenkins): on every PR, run `pip install -r requirements-dev.txt && pytest tests/ -v`. On merge to main, package the code (`zip src/*.py -o libs.zip`), upload to S3 (`aws s3 cp libs.zip s3://bucket/code/`), and optionally trigger an EMR step or Databricks job run. Key principle: tests must be fast (local SparkSession, no external I/O) — if CI takes 30 minutes, developers stop running it.

**Q: What is ruff and how does it differ from flake8/pylint?**
> `ruff` is an extremely fast Python linter written in Rust — 10-100× faster than flake8. It replaces flake8, isort, pyflakes, and many plugins in one tool. Run `ruff check .` to find issues, `ruff check --fix .` to auto-fix. Essential rules for data engineering: E (pycodestyle), F (pyflakes, catches undefined names), I (import sorting). Use `ruff format` as a black replacement for formatting.

**Q: What is the difference between unit tests and integration tests in CI?**
> Unit tests: test pure transformation functions with in-memory DataFrames — no external I/O, run in < 30 seconds. Run on every PR commit. Integration tests: test full pipeline stages reading/writing to test S3 buckets or local filesystem — slower (2-5 min). Run on PR review ready. End-to-end tests: full pipeline in a staging environment — slowest (30+ min). Run before production deploy.

In [ ]:
print("CI/CD for Spark — GitHub Actions patterns")
print("This notebook covers configuration files and workflow patterns.")

## Part 1: GitHub Actions Workflow

In [ ]:
print("""
.github/workflows/ci.yml — Full CI Pipeline:
══════════════════════════════════════════════════════════════

name: Spark ETL CI

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]

env:
  PYTHON_VERSION: '3.11'
  JAVA_VERSION: '17'

jobs:
  lint:
    name: Lint & Format
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - name: Cache pip
        uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: pip-${{ hashFiles('requirements-dev.txt') }}

      - run: pip install ruff mypy

      - name: Lint with ruff
        run: ruff check src/ tests/

      - name: Format check
        run: ruff format --check src/ tests/

      - name: Type check
        run: mypy src/ --ignore-missing-imports

  test:
    name: Unit Tests
    runs-on: ubuntu-latest
    needs: lint

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}

      - uses: actions/setup-java@v4
        with:
          java-version: ${{ env.JAVA_VERSION }}
          distribution: temurin

      - name: Cache pip
        uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: pip-${{ hashFiles('requirements-dev.txt') }}

      - run: pip install -r requirements-dev.txt

      - name: Run unit tests
        run: |
          pytest tests/unit/ -v \\
            --cov=src \\
            --cov-report=xml:coverage.xml \\
            --cov-fail-under=80

      - name: Upload coverage
        uses: codecov/codecov-action@v4
        with:
          file: coverage.xml

  deploy:
    name: Package & Deploy to S3
    runs-on: ubuntu-latest
    needs: test
    if: github.ref == 'refs/heads/main'   # only on main branch

    permissions:
      id-token: write   # OIDC auth to AWS (no stored credentials!)
      contents: read

    steps:
      - uses: actions/checkout@v4

      - name: Configure AWS credentials (OIDC)
        uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: arn:aws:iam::123456789:role/GitHubActionsDeployRole
          aws-region: us-east-1

      - name: Package src into zip
        run: |
          cd src && zip -r ../libs.zip . -x '__pycache__/*' -x '*.pyc'

      - name: Upload to S3
        run: |
          VERSION=$(git rev-parse --short HEAD)
          aws s3 cp libs.zip s3://my-bucket/code/$VERSION/libs.zip
          aws s3 cp libs.zip s3://my-bucket/code/latest/libs.zip

      - name: Tag release
        run: |
          echo "Deployed version: $(git rev-parse --short HEAD)"
══════════════════════════════════════════════════════════════
""")

## Part 2: ruff Configuration

In [ ]:
print("""
ruff.toml — Linting configuration:
══════════════════════════════════════════════════════════════

# ruff.toml (in project root)
line-length = 120
target-version = 'py311'

[lint]
select = [
    'E',    # pycodestyle errors
    'W',    # pycodestyle warnings
    'F',    # pyflakes (undefined names, unused imports)
    'I',    # isort (import ordering)
    'N',    # pep8-naming
    'UP',   # pyupgrade (modernize Python syntax)
    'B',    # flake8-bugbear (common bugs)
    'SIM',  # flake8-simplify
]
ignore = [
    'E501',  # line too long (ruff format handles this)
    'B008',  # do not perform function calls in default argument
]

[lint.per-file-ignores]
'tests/**/*.py' = ['S101']  # allow assert in tests

[format]
quote-style = 'double'
indent-style = 'space'

Running ruff:
  ruff check .                    # lint all files
  ruff check . --fix              # auto-fix safe issues
  ruff format .                   # format (like black)
  ruff check src/silver_transform.py  # single file

Common issues ruff catches for Spark code:
  F401: unused import (import F but never use it)
  F841: local variable assigned but never used (_tmp = df.count())
  E711: comparison to None (use 'is None' not '== None')
  B006: mutable default argument (def f(x=[]) → bug!)
  I001: imports not sorted
  UP006: use 'list' instead of 'List' (Python 3.9+)
══════════════════════════════════════════════════════════════
""")

## Part 3: Test Coverage & Quality Gates

In [ ]:
print("""
Test Coverage & Quality Gates:
══════════════════════════════════════════════════════════════

pytest.ini:
  [pytest]
  testpaths = tests
  python_files = test_*.py
  python_classes = Test*
  python_functions = test_*
  addopts = -v --tb=short

.coveragerc:
  [run]
  source = src
  omit =
      src/__init__.py
      src/main.py          # I/O entry point, test separately

  [report]
  fail_under = 80           # CI fails if coverage < 80%
  show_missing = True
  exclude_lines =
      pragma: no cover
      if __name__ == .__main__.

Quality gates — CI fails if:
  ruff check . → any linting errors
  ruff format --check . → any unformatted files
  pytest → any test failure
  pytest --cov-fail-under=80 → coverage < 80%
  mypy → any type errors

Makefile (developer convenience):
  .PHONY: test lint format check

  lint:
      ruff check src/ tests/
      mypy src/ --ignore-missing-imports

  format:
      ruff format src/ tests/
      ruff check --fix src/ tests/

  test:
      pytest tests/unit/ -v --cov=src --cov-report=term

  check: lint test   # run everything

  # Developers run: make check   before pushing
══════════════════════════════════════════════════════════════
""")

## Part 4: Deploying to EMR/Databricks from CI

In [ ]:
print("""
CD — Deploying Spark Code from GitHub Actions:
══════════════════════════════════════════════════════════════

# .github/workflows/deploy.yml
name: Deploy
on:
  push:
    branches: [main]

jobs:
  deploy-emr:
    runs-on: ubuntu-latest
    environment: production    # requires manual approval in GitHub

    steps:
      - uses: actions/checkout@v4

      - name: Configure AWS via OIDC (no static secrets)
        uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: arn:aws:iam::123:role/GHADeployRole
          aws-region: us-east-1

      - name: Package + upload
        run: |
          VERSION=$(git rev-parse --short HEAD)
          zip -r dist/code-$VERSION.zip src/
          aws s3 cp dist/code-$VERSION.zip \\
              s3://my-bucket/releases/$VERSION/code.zip

          # Update Airflow variable to point to new code version
          aws ssm put-parameter \\
              --name /airflow/code_version \\
              --value $VERSION \\
              --overwrite

  deploy-databricks:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Databricks CLI
        run: pip install databricks-cli

      - name: Upload to DBFS
        env:
          DATABRICKS_HOST:  ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
        run: |
          databricks fs cp src/silver_transform.py \\
              dbfs:/FileStore/scripts/silver_transform.py --overwrite

          # Trigger a job run with the new code
          databricks jobs run-now --job-id ${{ vars.DATABRICKS_JOB_ID }}

AWS OIDC Authentication (no stored secrets):
  Problem: storing AWS_ACCESS_KEY_ID in GitHub secrets = long-lived credential risk
  Solution: GitHub OIDC → AWS STS → temporary credentials (1 hour)

  Setup:
  1. Create GitHub OIDC provider in AWS IAM
  2. Create IAM Role with trust policy for github.com/your-org/your-repo
  3. GitHub Action uses 'id-token: write' permission
  4. aws-actions/configure-aws-credentials@v4 exchanges OIDC token for AWS credentials
  No static keys stored anywhere!
══════════════════════════════════════════════════════════════
""")

## Part 5: Integration Testing with pytest

In [ ]:
print("""
Integration Tests — Reading/Writing Files:
══════════════════════════════════════════════════════════════

# tests/integration/test_silver_pipeline.py
import pytest
import tempfile, os, shutil
from pyspark.sql import functions as F
from src.silver_transform import transform_silver

@pytest.fixture(scope='module')
def tmp_path():
    path = tempfile.mkdtemp()
    yield path
    shutil.rmtree(path)  # cleanup after module

class TestSilverPipeline:
    def test_reads_parquet_and_transforms(self, spark, tmp_path):
        # Arrange: write Bronze data
        bronze_path = os.path.join(tmp_path, 'bronze')
        spark.createDataFrame([
            ('O1', 'C1', 100.0, 'shipped',  '2024-01-15 10:00'),
            ('O2', None,  50.0, 'pending',  '2024-01-15 10:01'),
        ], ['order_id', 'customer_id', 'amount', 'status', 'event_time']) \
        .write.mode('overwrite').parquet(bronze_path)

        # Act: run the Silver transformation
        bronze_df = spark.read.parquet(bronze_path)
        silver_df = transform_silver(bronze_df)

        # Write Silver
        silver_path = os.path.join(tmp_path, 'silver')
        silver_df.write.mode('overwrite').parquet(silver_path)

        # Assert
        result = spark.read.parquet(silver_path)
        assert result.count() == 1  # null customer removed
        assert result.filter(F.col('order_id') == 'O1').count() == 1

    def test_idempotent_write(self, spark, tmp_path):
        silver_path = os.path.join(tmp_path, 'silver_idempotent')
        df = spark.createDataFrame([('O1','C1',100.0,'shipped')],
                                   ['order_id','customer_id','amount','status'])

        # Run twice
        df.write.mode('overwrite').parquet(silver_path)
        df.write.mode('overwrite').parquet(silver_path)

        assert spark.read.parquet(silver_path).count() == 1  # not doubled

Running integration tests separately:
  pytest tests/unit/      -v                 # fast (~30s)
  pytest tests/integration/ -v --slow        # slower (~5min)

  In CI:
    on PR:   run unit tests only
    on merge: run unit + integration
    before prod deploy: run all including E2E
══════════════════════════════════════════════════════════════
""")

## Exercises

1. Write a complete `.github/workflows/ci.yml` for a PySpark ETL project with two jobs: `lint` (ruff + mypy) and `test` (pytest with coverage, requires Java 17 + Python 3.11). Make `test` depend on `lint`.
2. Write a `ruff.toml` for a data engineering project. Enable E, F, I, B rules. Ignore E501 (line length). Allow `assert` in test files. Set `line-length=100` and `target-version=py311`.
3. Write a `deploy.yml` GitHub Actions workflow that: runs only on push to `main`, packages `src/` into a zip, uploads to S3 using OIDC authentication (no stored keys), and prints the deployed version (git short hash).
4. What is GitHub OIDC authentication and why is it better than storing `AWS_ACCESS_KEY_ID` in GitHub secrets? Walk through the setup steps.
5. Your CI pipeline takes 12 minutes: 2 min lint, 10 min tests. The 10 minutes is because the SparkSession starts from scratch for each test file (no session fixture). How do you fix this and what speedup would you expect?